In [1]:
# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements.  See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership.  The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License.  You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied.  See the License for the
# specific language governing permissions and limitations
# under the License.

In [ ]:
!pip install apache-burr

# Stopping a streaming action early

When you stop reading a stream before it ends, Burr runs the action's cleanup right away, then reports the step as failed (so resuming re-runs it), and doesn't update the state. **Sync code just `break`s. Async code `break`s inside `async with contextlib.aclosing(...)`.**

Each demo below prints a timeline of what happened, in order. The code is in [application.py](application.py), and the explanation is in [README.md](README.md).

In [3]:
import asyncio
import logging

import application as demo

logging.basicConfig(level=logging.ERROR)

## Sync: just `break`

In [4]:
demo.sync_break_demo()


=== Sync: break
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: after the loop
   caller: get() -> result=None, 'response' in state=False


In [5]:
demo.sync_iterate_break_demo()


=== Sync: break out of stream_iterate
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: after the loop


## Async: `break` inside `contextlib.aclosing`

In [6]:
await demo.async_aclosing_demo()


=== Async: break inside contextlib.aclosing(container)
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: left the `async with` block


In [7]:
await demo.async_iterate_aclosing_demo()


=== Async: break inside contextlib.aclosing(app.astream_iterate(...))
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: left the `async with` block


Without `aclosing`, Python leaves the async cleanup to the event loop, to run at some later point:

In [8]:
await demo.async_break_without_aclose_demo()


=== Async without aclosing: cleanup is left to the event loop (avoid)
   respond: open LLM connection
   caller: after the loop -- nothing has been cleaned up yet
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: some time later


## Web servers: a client disconnect cancels the consuming task

The hook receives `CancelledError`. See [server.py](server.py) for the FastAPI version.

In [9]:
await demo.client_disconnect_demo()


=== Web server: client disconnect cancels the consuming task
   respond: open LLM connection
   server: sent 'Burr '
   server: sent 'streams '
   server: client disconnected -> task.cancel()
   respond: close LLM connection
   post_run_step: respond failed with CancelledError()


## Persistence: a stopped step is saved as `failed`, and resuming re-runs it

In [10]:
demo.resume_after_stop_demo()


=== Persistence: a stopped step is 'failed' and is re-run on resume
   respond: open LLM connection
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   persister: last step saved as 'failed'
   resume: next action is 'respond' (it runs again)


## Cleanup errors: reported to hooks, and raised by `close()`

In [11]:
demo.cleanup_error_demo()


=== Cleanup errors: reported to hooks, and raised by close()
   respond: open LLM connection
   respond: close LLM connection
   post_run_step: respond failed with ConnectionError('LLM connection did not close cleanly')
   caller: close() raised ConnectionError('LLM connection did not close cleanly')
